# core

> Async Python client for Google Colab runtimes, kernels, files, and Drive transfers

- skip_exec: true

In [ ]:
#| default_exp core

This notebook is the literate source for `fastcolab.core`, building the library from raw REST calls into a small async client.

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import asyncio, base64, time, uuid
from pathlib import Path
from urllib.parse import quote

from fastcore.utils import *
from fastgws.core import GWSApi, oauth_creds
from fasttransport.errors import APIError
from jupyasyncclient.core import JupyAsyncKernelClient
from jupyasyncclient.files import JupyAsyncFilesClient
from jupyasyncclient.http import http_client

In [ ]:
import json, os

## Connecting to the Colab REST API

Google splits Colab's REST API across two discovery documents under `colaboratory.googleapis.com`. The `v1beta` document handles subscriptions, eligible runtime specs, and runtime lifecycle, while `v1` handles long-running operations. Because `colaboratory` is absent from Google's central discovery directory, `fastgws` loads each document directly by URL.

In [ ]:
#| export
COLAB_SCOPE = 'https://www.googleapis.com/auth/colaboratory'
DRIVE_SCOPE = 'https://www.googleapis.com/auth/drive'
COLAB_V1BETA_URL = 'https://colaboratory.googleapis.com/$discovery/rest?version=v1beta'
COLAB_V1_URL = 'https://colaboratory.googleapis.com/$discovery/rest?version=v1'
PROXY_HEADER = 'X-Colab-Runtime-Proxy-Token'
CHUNK_SIZE = 32 * 1024 * 1024

Authenticate with `oauth_creds` from `fastgws` using the Google account holding your Colab or Google AI Ultra subscription. Requesting both `COLAB_SCOPE` and `DRIVE_SCOPE` lets the same credentials talk to Colab and Google Drive with no browser prompts.

In [ ]:
acct = os.environ['FASTCOLAB_ACCOUNT']
creds = await oauth_creds(account=acct, scopes=[COLAB_SCOPE, DRIVE_SCOPE])
v1beta = await GWSApi.from_discovery_url(COLAB_V1BETA_URL, creds=creds)
v1 = await GWSApi.from_discovery_url(COLAB_V1_URL, creds=creds)

Both read-only calls below run without provisioning a virtual machine or burning compute units.

In [ ]:
sub = await v1beta.v1beta.get_subscription(name='subscription')
sub

```python
{'name': 'subscription', 'tier': 'SUBSCRIPTION_TIER_PRO_PLUS'}
```

In [ ]:
raw_specs = await v1beta.runtimespecs.list()
raw_specs['runtimeSpecs'][:4]

[{'eligible': True, 'key': {'variant': 'VARIANT_CPU', 'accelerator': 'NONE', 'shape': 'SHAPE_HIGHMEM'}}, {'eligible': True, 'key': {'variant': 'VARIANT_CPU', 'accelerator': 'NONE', 'shape': 'SHAPE_STANDARD'}}, {'eligible': True, 'key': {'variant': 'VARIANT_GPU', 'accelerator': 'A100', 'shape': 'SHAPE_HIGHMEM'}}, {'eligible': True, 'key': {'variant': 'VARIANT_GPU', 'accelerator': 'T4', 'shape': 'SHAPE_HIGHMEM'}}]

## Choosing hardware

Google identifies each runtime by `variant` (`VARIANT_CPU`, `VARIANT_GPU`, or `VARIANT_TPU`), `accelerator`, and `shape` (`SHAPE_STANDARD` or `SHAPE_HIGHMEM`). Only CPU, T4, and A100 accept `SHAPE_HIGHMEM`. L4, G4, H100, and both TPUs already allocate high-memory hosts under `SHAPE_STANDARD`.

High-RAM gives CPU and T4 runtimes four times the memory and CPU cores for roughly 0.1 to 0.3 extra compute units per hour. `mk_spec` turns it on by default whenever the hardware supports a choice. Passing a raw dict skips the lookup table so new Google hardware works immediately.

In [ ]:
#| export
HW_SPECS = dict(cpu=('VARIANT_CPU', 'NONE', True), t4=('VARIANT_GPU', 'T4', True), l4=('VARIANT_GPU', 'L4', False), a100=('VARIANT_GPU', 'A100', True),
    g4=('VARIANT_GPU', 'G4', False), h100=('VARIANT_GPU', 'H100', False), v5e1=('VARIANT_TPU', 'V5E1', False), v6e1=('VARIANT_TPU', 'V6E1', False))

def mk_spec(
    hw:str|dict='cpu', # Hardware alias (`'cpu'`, `'t4'`, `'l4'`, `'g4'`, `'h100'`, `'v6e1'`, ...) or raw spec dict
    high_mem:bool=True, # Request `SHAPE_HIGHMEM` when the hardware supports a choice?
)->dict:
    "Build a Colab `runtimeSpec` dict from a short hardware alias or pass through a raw dict."
    if isinstance(hw, dict): return hw
    key = hw.lower().replace('-', '')
    if key not in HW_SPECS: raise ValueError(f'Unknown hardware {hw!r}; expected one of {sorted(HW_SPECS)}')
    variant, acc, has_hm = HW_SPECS[key]
    return dict(variant=variant, accelerator=acc, shape='SHAPE_HIGHMEM' if (high_mem and has_hm) else 'SHAPE_STANDARD')

In [ ]:
test_eq(mk_spec('cpu'), dict(variant='VARIANT_CPU', accelerator='NONE', shape='SHAPE_HIGHMEM'))
test_eq(mk_spec('cpu', high_mem=False), dict(variant='VARIANT_CPU', accelerator='NONE', shape='SHAPE_STANDARD'))
test_eq(mk_spec('g4'), dict(variant='VARIANT_GPU', accelerator='G4', shape='SHAPE_STANDARD'))
test_eq(mk_spec('v6e1'), dict(variant='VARIANT_TPU', accelerator='V6E1', shape='SHAPE_STANDARD'))
mk_spec('g4')

{'variant': 'VARIANT_GPU', 'accelerator': 'G4', 'shape': 'SHAPE_STANDARD'}

## The `Colab` client

This container holds the `v1beta` and `v1` discovery clients plus an optional `fastgws` Drive client. We define `__init__` and `connect` first, then `@patch` each operation onto it beside a live example.

In [ ]:
#| export
class Colab:
    "Async client for Google Colab runtimes and optional Google Drive transfers."
    def __init__(
        self,
        v1beta, # `GWSApi` client for Colab `v1beta`
        v1, # `GWSApi` client for Colab `v1` operations
        creds=None, # OAuth credentials shared with Drive
        drive=None, # Optional `GWSApi('drive', ...)` client
    ): store_attr()

    def __repr__(self): return f'Colab(drive={self.drive is not None})'

    @classmethod
    async def connect(
        cls,
        account:str=None, # Google account email stored by `gclientid`
        creds=None, # Existing `google-auth` credentials (skips `oauth_creds` when given)
        scopes:list=None, # OAuth scopes to request (defaults to Colab + Drive when `drive=True`)
        drive:bool=True, # Also attach a `fastgws` Drive client?
        **kwargs,
    ):
        "Authenticate and load both Colab discovery clients."
        if creds is None:
            sc = scopes or ([COLAB_SCOPE, DRIVE_SCOPE] if drive else [COLAB_SCOPE])
            creds = await oauth_creds(account=account, scopes=sc, **kwargs)
        v1b, v1 = await asyncio.gather(*(GWSApi.from_discovery_url(u, creds=creds) for u in (COLAB_V1BETA_URL, COLAB_V1_URL)))
        return cls(v1b, v1, creds=creds, drive=GWSApi('drive', creds=creds) if drive else None)

Create a `Colab` client reusing the credentials we already loaded:

In [ ]:
c = await Colab.connect(creds=creds)
c

Colab(drive=True)

`subscription` and `specs` query account entitlements and available hardware without allocating anything.

In [ ]:
#| export
@patch
async def subscription(self:Colab)->str:
    "Return the Colab subscription tier string for the authenticated user."
    return (await self.v1beta.v1beta.get_subscription(name='subscription')).get('tier', '')

In [ ]:
tier = await c.subscription()
assert tier
tier

'SUBSCRIPTION_TIER_PRO_PLUS'

`specs` lists the runtime specs returned by `runtimespecs.list`, filtered to `eligible=True` by default.

In [ ]:
#| export
@patch
async def specs(
    self:Colab,
    eligible_only:bool=True, # Filter to specs the current account can allocate?
)->L:
    "List available Colab runtime specs as `AttrDict` entries."
    res = await self.v1beta.runtimespecs.list()
    items = L(res.get('runtimeSpecs', [])).map(lambda s: AttrDict(eligible=s.get('eligible', False), **s['key']))
    return items.filter(lambda s: s.eligible) if eligible_only else items

Inspect the first three eligible specs on our account:

In [ ]:
avail = await c.specs()
assert len(avail) > 0
avail[:3]

[{'eligible': True, 'variant': 'VARIANT_CPU', 'accelerator': 'NONE', 'shape': 'SHAPE_HIGHMEM'}, {'eligible': True, 'variant': 'VARIANT_CPU', 'accelerator': 'NONE', 'shape': 'SHAPE_STANDARD'}, {'eligible': True, 'variant': 'VARIANT_GPU', 'accelerator': 'A100', 'shape': 'SHAPE_HIGHMEM'}]

## Starting and listing runtimes

`ColabRuntime` wraps one live Colab virtual machine, exposing its `name`, hardware `spec`, Jupyter proxy `url`, `token`, and `headers`. Both `__repr__` and `_repr_markdown_` keep the proxy token out of printed output.

In [ ]:
#| export
def _rt_name(name:str)->str: return name if name.startswith('runtimes/') else f'runtimes/{name}'
def _clean_path(path:str|Path)->str: return str(path).lstrip('/')

class ColabRuntime:
    "Handle to an active Google Colab runtime and its Jupyter proxy."
    def __init__(
        self,
        colab:Colab, # Parent `Colab` client
        data:dict, # Runtime resource dict returned by `v1beta.runtimes`
        owned:bool=False, # Delete the runtime automatically on `async with` exit?
    ):
        self.colab,self.data,self.owned,self._kc = colab,AttrDict(data),owned,None

    @property
    def name(self)->str: return self.data['name']
    @property
    def id(self)->str: return self.name.split('/')[-1]
    @property
    def spec(self)->dict: return self.data.get('runtimeSpec', {})
    @property
    def conn(self)->dict: return self.data.get('connectionInfo', {})
    @property
    def url(self)->str: return self.conn['url']
    @property
    def token(self)->str: return self.conn['token']
    @property
    def expire_time(self)->str: return self.conn.get('expireTime', '')
    @property
    def headers(self)->dict: return {PROXY_HEADER: self.token}

    def __repr__(self): return f"ColabRuntime({self.id!r}, {self.spec.get('accelerator')}, {self.spec.get('shape')})"
    def _repr_markdown_(self):
        s = self.spec
        return f"`ColabRuntime({self.id})` ({s.get('variant')}, {s.get('accelerator')}, {s.get('shape')}), expires `{self.expire_time}`"

Creating a runtime returns a Google long-running `Operation` instead of the finished runtime. `_wait_op` waits on `v1.operations.wait` until `done` is true. If `wait` errors or returns early, it falls back to `v1.operations.get`. Any backend error raises immediately.

In [ ]:
#| export
async def _wait_op(ops, op:dict, timeout:float=180, poll:float=2.0)->dict:
    "Wait for a Colab `v1.operations` resource to finish and return its response payload."
    deadline = time.monotonic() + timeout
    while not op.get('done'):
        rem = deadline - time.monotonic()
        if rem <= 0: raise TimeoutError(f"Colab operation {op.get('name')} timed out after {timeout}s")
        t0 = time.monotonic()
        try: op = await ops.operations.wait(name=op['name'], timeout=f'{max(1, int(min(30, rem)))}s')
        except APIError: op = await ops.operations.get(name=op['name'])
        if not op.get('done') and (time.monotonic() - t0) < poll: await asyncio.sleep(poll)
    if op.get('error'): raise RuntimeError(f"Colab operation failed: {op['error']}")
    return op.get('response', op)

`start` attaches a fresh UUID `request_id` for idempotent retries and waits for provisioning to finish. If the operation response omits `connectionInfo`, it fetches the runtime with `v1beta.runtimes.get`.

In [ ]:
#| export
@patch
async def start(
    self:Colab,
    hw:str|dict='cpu', # Hardware alias or raw `runtimeSpec` dict
    high_mem:bool=True, # Request `SHAPE_HIGHMEM` when supported?
    runtime_id:str=None, # Optional RFC-1034 runtime ID
    version:str=None, # Optional Colab runtime image version
    timeout:float=180, # Seconds to wait for provisioning
)->ColabRuntime:
    "Provision a Colab runtime, wait until it is ready, and return a `ColabRuntime`."
    kw = dict(runtime_spec=mk_spec(hw, high_mem=high_mem), request_id=str(uuid.uuid4()))
    if runtime_id: kw['runtime_id'] = runtime_id
    if version: kw['version'] = version
    data = await _wait_op(self.v1, await self.v1beta.runtimes.create(**kw), timeout=timeout)
    if 'connectionInfo' not in data and 'name' in data: data = await self.v1beta.runtimes.get(name=data['name'])
    return ColabRuntime(self, data, owned=True)

Start a live high-memory CPU runtime now. We reuse this single server through the rest of the notebook for computations, file transfers, Google Drive sync, and detached job polling. The final section stops it.

In [ ]:
rt = await c.start('cpu')
rt

`ColabRuntime(r-1a19e1fe-152d-4898-b5fb-0dcc53bf4e8d)` (VARIANT_CPU, NONE, SHAPE_HIGHMEM), expires `2026-11-22T05:36:30.563937260Z`

`runtimes`, `get`, and `delete` inspect and manage existing runtimes on the account.

In [ ]:
#| export
@patch
async def runtimes(self:Colab)->L:
    "List currently active Colab runtimes."
    return L((await self.v1beta.runtimes.list()).get('runtimes', [])).map(lambda d: ColabRuntime(self, d))

@patch
async def get(
    self:Colab,
    name:str, # Full resource name (`runtimes/{id}`) or bare runtime ID
)->ColabRuntime:
    "Fetch one active runtime with fresh `connectionInfo`."
    return ColabRuntime(self, await self.v1beta.runtimes.get(name=_rt_name(name)))

@patch
async def delete(
    self:Colab,
    name:str, # Full resource name (`runtimes/{id}`) or bare runtime ID
    allow_missing:bool=True, # Succeed silently if the runtime is already gone?
):
    "Delete a Colab runtime assignment."
    return await self.v1beta.runtimes.delete(name=_rt_name(name), allow_missing=allow_missing)

In [ ]:
active = await c.runtimes()
assert any(r.name == rt.name for r in active)
active

[ColabRuntime('r-1a19e1fe-152d-4898-b5fb-0dcc53bf4e8d', NONE, SHAPE_HIGHMEM)]

## Refreshing tokens and running code

Colab proxy tokens expire about one hour after issue and return HTTP 404 once stale. `refresh` fetches fresh `connectionInfo` via `v1beta.runtimes.get` and returns the updated header dict. That matches the `refresh_headers` callback expected by `JupyAsyncKernelClient`.

In [ ]:
#| export
@patch
async def refresh(self:ColabRuntime)->dict:
    "Refresh `connectionInfo` via `v1beta.runtimes.get` and return updated proxy headers."
    self.data = AttrDict(await self.colab.v1beta.runtimes.get(name=self.name))
    return self.headers

In [ ]:
hdrs = await rt.refresh()
assert PROXY_HEADER in hdrs
list(hdrs)

['X-Colab-Runtime-Proxy-Token']

`outs_text` pulls plain text from Jupyter nbformat output dicts (stdout, stderr, expression results, and tracebacks).

In [ ]:
#| export
def outs_text(outs:list)->str:
    "Join stdout, stderr, execute_result text, and error tracebacks from nbformat outputs."
    parts = []
    for o in outs:
        t = o.get('output_type')
        if t == 'stream': parts.append(o.get('text', ''))
        elif t in ('execute_result', 'display_data'):
            txt = o.get('data', {}).get('text/plain')
            if txt is not None: parts.append(f'{txt}\n')
        elif t == 'error': parts.append('\n'.join(o.get('traceback', [])) or f"{o.get('ename')}: {o.get('evalue')}\n")
    return ''.join(parts)

Verify plain stream extraction:

In [ ]:
test_eq(outs_text([dict(output_type='stream', text='hello\n')]), 'hello\n')

`kernel` opens a `JupyAsyncKernelClient` wired with `X-Colab-Runtime-Proxy-Token` and automatic `refresh_headers`. Passing `kernel_id` attaches to an existing kernel without taking ownership. `exec` reuses a lazy default kernel on `rt` and returns rendered nbformat outputs.

In [ ]:
#| export
@patch
async def kernel(
    self:ColabRuntime,
    kernel_id:str=None, # Existing kernel ID to attach to, or `None` to start a new kernel
    timeout:float=60, # Connection timeout in seconds
)->JupyAsyncKernelClient:
    "Connect a `JupyAsyncKernelClient` to this runtime's Jupyter proxy."
    return await JupyAsyncKernelClient.connect(
        self.url, kernel=kernel_id, headers=self.headers, refresh_headers=self.refresh, timeout=timeout)

@patch
async def exec(
    self:ColabRuntime,
    code:str, # Python source to execute on the runtime's default kernel
    timeout:float=None, # Execution timeout in seconds (`None` waits indefinitely)
)->list:
    "Execute `code` on `rt`'s persistent kernel and return nbformat outputs."
    if self._kc is None or not self._kc.channels_running: self._kc = await self.kernel()
    return await self._kc.exec_outs(code, timeout=timeout)

Let's inspect the remote host specs and run a small computation. Because `rt.exec` reuses the same remote kernel, variables set in one call stay in memory for the next.

In [ ]:
outs = await rt.exec('''
import os, platform
with open('/proc/meminfo') as f:
    mem_gb = next(int(l.split()[1]) for l in f if l.startswith('MemTotal:')) / 1e6
xs = [i * i for i in range(1, 101)]
print(f"Python {platform.python_version()} | {os.cpu_count()} cores | {mem_gb:.1f} GB RAM")
sum(xs)
''')
txt = outs_text(outs)
assert '338350' in txt
print(txt)

Python 3.13.18 | 8 cores | 53.5 GB RAM
338350



In [ ]:
outs2 = await rt.exec('len(xs), xs[-1]')
txt2 = outs_text(outs2)
assert '(100, 10000)' in txt2
txt2.strip()

'(100, 10000)'

## Transferring files

Colab's proxy exposes two file endpoints. `/api/contents/{path}` speaks the Jupyter contents API (wrapped by `JupyAsyncFilesClient`), requiring chunked `PUT` uploads above 50 MB. `/files/{path}` serves raw bytes with HTTP `Range` support for binary downloads and live log tailing.

`files` returns a `JupyAsyncFilesClient` pointed at the runtime. `upload` builds on it to create missing parent folders one level at a time and split files larger than `chunk_size` (32 MB by default).

In [ ]:
#| export
@patch
def files(self:ColabRuntime)->JupyAsyncFilesClient:
    "Return a `JupyAsyncFilesClient` for this runtime's Jupyter contents API."
    return JupyAsyncFilesClient(self.url, headers=self.headers)

async def _ensure_parent_dirs(rt:ColabRuntime, remote_path:str):
    parts = _clean_path(remote_path).split('/')[:-1]
    fc, cur = rt.files(), ''
    for p in parts:
        cur = f'{cur}/{p}' if cur else p
        if cur == 'content': continue
        try: await fc.put(cur, type='directory')
        except APIError as e:
            if e.status_code not in (400, 409): raise

List the default `/content` directory on our live runtime:

In [ ]:
[e['name'] for e in await rt.files().listing('content')]

['.config', 'sample_data']

`upload` creates missing parent directories and automatically chunks files larger than `chunk_size`.

In [ ]:
#| export
@patch
async def upload(
    self:ColabRuntime,
    src:str|Path|bytes, # Local file path (`Path`) or raw `bytes`/`str` content to upload
    remote_path:str, # Destination path on the runtime (e.g. `'content/data/file.csv'`)
    chunk_size:int=CHUNK_SIZE, # Chunk size in bytes for multi-part uploads (must stay under 50 MB)
):
    "Upload bytes, text, or a local file to `remote_path`, creating parent directories and chunking large files."
    rpath = _clean_path(remote_path)
    await _ensure_parent_dirs(self, rpath)
    data = src.read_bytes() if isinstance(src, Path) else (src.encode('utf-8') if isinstance(src, str) else bytes(src))
    fc = self.files()
    if len(data) <= chunk_size: return await fc.put(rpath, type='file', format='base64', content=base64.b64encode(data).decode('ascii'))
    n, res = (len(data) + chunk_size - 1) // chunk_size, None
    for i in range(n):
        piece = base64.b64encode(data[i*chunk_size:(i+1)*chunk_size]).decode('ascii')
        res = await fc.put(rpath, type='file', format='base64', content=piece, chunk=-1 if i == n - 1 else i + 1)
    return res

Upload a small text file into a new subdirectory and read it back through `files()`:

In [ ]:
await rt.upload('line 1\nline 2\n', 'content/scratch/hello.txt')
await rt.files().read('content/scratch/hello.txt')

'line 1\nline 2\n'

`download` reads raw bytes from `/files/{path}` with an optional byte `offset`. On HTTP 404 it refreshes the proxy token once and retries. On HTTP 416 (when the file has not grown past `offset`) it returns empty bytes. `tail` wraps `download` for UTF-8 log files.

In [ ]:
@patch
async def download(
    self:ColabRuntime,
    remote_path:str, # Source path on the runtime (e.g. `'content/train.log'`)
    local_path:str|Path=None, # Optional local path to write the downloaded bytes to
    offset:int=0, # Byte offset for HTTP `Range` reads
)->bytes|Path:
    "Download raw bytes from `/files/{remote_path}`, supporting HTTP `Range` offsets and automatic token refresh."
    rpath = quote(_clean_path(remote_path), safe='/')
    for attempt in range(2):
        hdrs = dict(self.headers, Range=f'bytes={offset}-') if offset > 0 else dict(self.headers)
        async with http_client(self.url, headers=hdrs, timeout=60.0) as hc: resp = await hc.get(f'/files/{rpath}')
        if resp.status_code == 404 and attempt == 0:
            await self.refresh()
            continue
        if resp.status_code == 416:
            data = b''
            break
        if resp.status_code >= 400:
            raise FileNotFoundError(remote_path) if resp.status_code == 404 else RuntimeError(f'GET /files/{rpath} failed: HTTP {resp.status_code}')
        data = resp.content[offset:] if (offset > 0 and resp.status_code == 200) else resp.content
        break
    if local_path is None: return data
    p = Path(local_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(data)
    return p

In [ ]:
@patch
async def tail(
    self:ColabRuntime,
    remote_path:str, # Log file path on the runtime
    offset:int=0, # Byte offset returned by the previous `tail` call
)->tuple[str, int]:
    "Read newly appended UTF-8 text from `remote_path` starting at `offset`, returning `(text, new_offset)`."
    raw = await self.download(remote_path, offset=offset)
    return raw.decode('utf-8', errors='replace'), offset + len(raw)

In [ ]:
raw = await rt.download('content/scratch/hello.txt')
test_eq(raw, b'line 1\nline 2\n')
chunk1, off = await rt.tail('content/scratch/hello.txt', offset=0)
await rt.exec("open('/content/scratch/hello.txt', 'a').write('line 3\\n')")
chunk2, off = await rt.tail('content/scratch/hello.txt', offset=off)
test_eq((chunk1, chunk2), ('line 1\nline 2\n', 'line 3\n'))
chunk2

'line 3\n'

## Importing from and saving to Google Drive

Native `google.colab.drive.mount` needs interactive browser consent and Colab's older tunnel endpoint. Because `Colab.connect` already holds a `fastgws` Drive client on the same OAuth credentials, `from_drive` and `to_drive` copy files directly between Google Drive and Colab without browser popups.

`_drive_folder_id` resolves a folder name, a slash-separated path like `'datasets/demo'`, or a raw Drive folder ID. Pass `create=True` to create missing folders along the path.

In [ ]:
#| export
_FOLDER_MIME = 'application/vnd.google-apps.folder'
def _esc_q(s:str)->str: return s.replace('\\', '\\\\').replace("'", "\\'")

async def _drive_folder_id(
    drive, # `GWSApi('drive', ...)` client
    folder:str, # Folder name, slash-separated path (e.g. `'datasets/demo'`), or Drive folder ID
    create:bool=False, # Create missing folders along the path?
)->str:
    "Resolve a Google Drive folder path or name to its Drive file ID."
    parent, parts = 'root', [p for p in folder.strip('/').split('/') if p]
    for name in parts:
        q = f"name = '{_esc_q(name)}' and mimeType = '{_FOLDER_MIME}' and '{parent}' in parents and trashed = false"
        files = (await drive.files.list(q=q, fields='files(id, name)', page_size=10)).get('files', [])
        if files: parent = files[0]['id']
        elif len(parts) == 1 and not create:
            meta = await drive.files.get(file_id=name, fields='id, mimeType, trashed')
            if meta.get('mimeType') == _FOLDER_MIME and not meta.get('trashed'): return meta['id']
            raise FileNotFoundError(f'Drive folder not found: {folder!r}')
        elif create: parent = (await drive.files.create(name=name, mime_type=_FOLDER_MIME, parents=[parent], fields='id'))['id']
        else: raise FileNotFoundError(f'Drive folder not found: {folder!r}')
    return parent

Create a temporary Google Drive folder and upload a tiny CSV into it. We will import that file into our Colab runtime next.

In [ ]:
demo_folder = f'fastcolab-nb-demo-{uuid.uuid4().hex[:8]}'
demo_fid = await _drive_folder_id(c.drive, demo_folder, create=True)
csv_bytes = b'city,temp_c\nSydney,22\nBrisbane,26\nMelbourne,18\n'
await c.drive.files.upload(media=csv_bytes, name='readings.csv', parents=[demo_fid], fields='id, name')

```python
{'id': '1LlveMseDnT66q0YMQoO514WcMGUOh0te', 'name': 'readings.csv'}
```

`from_drive` lists non-trashed files in the Drive folder, downloads each file's bytes, and uploads them into `remote_dir` on the runtime.

In [ ]:
#| export
@patch
async def from_drive(
    self:ColabRuntime,
    folder:str, # Google Drive folder name, path, or folder ID
    remote_dir:str='content/data', # Destination directory on the Colab runtime
    names:list=None, # Optional subset of filenames to copy (defaults to all non-folder files)
)->L:
    "Copy files from a Google Drive folder onto this Colab runtime, returning uploaded remote paths."
    drive = self.colab.drive
    if drive is None: raise RuntimeError('Drive client is disabled; pass drive=True to Colab.connect()')
    fid = await _drive_folder_id(drive, folder, create=False)
    q = f"'{fid}' in parents and mimeType != '{_FOLDER_MIME}' and trashed = false"
    entries = (await drive.files.list(q=q, fields='files(id, name, mimeType)', page_size=100)).get('files', [])
    if names is not None: entries = [e for e in entries if e['name'] in set(names)]
    dest, uploaded = _clean_path(remote_dir).rstrip('/'), L()
    for e in entries:
        resp = await drive.files.get(file_id=e['id'], alt='media', raw_=True)
        rpath = f"{dest}/{e['name']}"
        await self.upload(resp.content, rpath)
        uploaded.append(rpath)
    return uploaded

Pull the demo folder into `/content/data`:

In [ ]:
imported = await rt.from_drive(demo_folder, 'content/data')
test_eq(imported, ['content/data/readings.csv'])
imported

['content/data/readings.csv']

Now let's process `/content/data/readings.csv` on the Colab server and write `/content/data/summary.json`, then save that result back to our Drive folder with `to_drive`.

In [ ]:
#| export
@patch
async def to_drive(
    self:ColabRuntime,
    remote_path:str, # File path on the Colab runtime to save
    folder:str, # Destination Google Drive folder name, path, or folder ID
    name:str=None, # Filename in Drive (defaults to the basename of `remote_path`)
    create_folder:bool=True, # Create the destination Drive folder if missing?
)->dict:
    "Copy a file from this Colab runtime into a Google Drive folder, updating in place if it already exists."
    drive = self.colab.drive
    if drive is None: raise RuntimeError('Drive client is disabled; pass drive=True to Colab.connect()')
    fid = await _drive_folder_id(drive, folder, create=create_folder)
    fname, data = name or Path(remote_path).name, await self.download(remote_path)
    q = f"name = '{_esc_q(fname)}' and '{fid}' in parents and trashed = false"
    existing = (await drive.files.list(q=q, fields='files(id, name)', page_size=5)).get('files', [])
    if existing: return await drive.files.update_media(file_id=existing[0]['id'], media=data, fields='id, name')
    return await drive.files.upload(media=data, name=fname, parents=[fid], fields='id, name')

In [ ]:
await rt.exec('''
import csv, json
with open('/content/data/readings.csv') as f: rows = list(csv.DictReader(f))
temps = [float(r['temp_c']) for r in rows]
summary = dict(count=len(temps), mean=sum(temps)/len(temps), min=min(temps), max=max(temps))
with open('/content/data/summary.json', 'w') as f: json.dump(summary, f)
summary
''')
saved = await rt.to_drive('content/data/summary.json', demo_folder)
saved

```python
{'id': '1hAcPv-5zc8sHLAU9nYrAyJlSOff-VOWI', 'name': 'summary.json'}
```

Read `summary.json` directly back from Google Drive to verify the saved file, then remove our temporary demo folder.

In [ ]:
resp = await c.drive.files.get(file_id=saved['id'], alt='media', raw_=True)
result = json.loads(resp.content)
test_eq(result, dict(count=3, mean=22.0, min=18.0, max=26.0))
await c.drive.files.delete(file_id=demo_fid)
result

{'count': 3, 'mean': 22.0, 'min': 18.0, 'max': 26.0}

## Running long detached jobs

Colab's ~90-minute idle timeout checks whether the Jupyter kernel is busy. Spawning a `nohup` background process and letting the cell exit leaves the kernel idle, risking teardown.

`launch` avoids that by running your code inside a dedicated kernel cell. A small wrapper redirects stdout and stderr to a log file and writes the exit code to a second file when finished. Once the kernel reports `busy`, `launch` closes the local WebSocket (`aclose()`) while the remote cell keeps running.

`ColabJob` tracks that detached run over plain HTTP. It refreshes expired proxy tokens automatically and reads new log bytes via HTTP `Range`.

In [ ]:
#| export
def _job_wrapper(code:str, log_path:str, exit_path:str)->str:
    return f'''import sys, traceback
_lf = open({log_path!r}, 'w', buffering=1)
_o, _e = sys.stdout, sys.stderr
sys.stdout = sys.stderr = _lf
_rc = 0
try: exec({code!r}, {{'__name__': '__main__'}})
except SystemExit as _ex: _rc = _ex.code if isinstance(_ex.code, int) else (1 if _ex.code else 0)
except BaseException:
    traceback.print_exc()
    _rc = 1
finally:
    sys.stdout, sys.stderr = _o, _e
    _lf.close()
    open({exit_path!r}, 'w').write(str(_rc))
'''

class ColabJob:
    "Handle to a detached cell running inside a Colab kernel, polled over HTTP."
    def __init__(
        self,
        rt:ColabRuntime, # Runtime hosting the job
        kernel_id:str, # Remote Jupyter kernel ID executing the job
        log:str, # Log file path on the runtime (without leading slash)
        exit_file:str, # Exit-code file path on the runtime (without leading slash)
    ):
        store_attr()
        self.offset = 0

    def __repr__(self): return f'ColabJob({self.kernel_id!r}, log={self.log!r})'

Check how `ColabJob` renders before launching a real run:

In [ ]:
ColabJob(rt, 'k-demo', 'content/job.log', 'content/job.exit')

ColabJob('k-demo', log='content/job.log')

`launch` clears any stale log or exit files and starts a dedicated kernel. Once the kernel sends `status: busy`, it closes the local WebSocket while the cell keeps running on Colab.

In [ ]:
#| export
@patch
async def launch(
    self:ColabRuntime,
    code:str, # Python source to execute in a dedicated detached kernel
    log:str='content/job.log', # Remote log file path
    exit_file:str='content/job.exit', # Remote exit-code file path
)->ColabJob:
    "Submit `code` to a dedicated kernel, close the local WebSocket once busy, and return a `ColabJob`."
    log_rel, exit_rel = _clean_path(log), _clean_path(exit_file)
    await _ensure_parent_dirs(self, log_rel)
    await _ensure_parent_dirs(self, exit_rel)
    fc = self.files()
    for p in (log_rel, exit_rel):
        try: await fc.delete(p)
        except APIError: pass
    kc = await self.kernel()
    gen = kc.run(_job_wrapper(code, '/' + log_rel, '/' + exit_rel))
    try:
        async for msg in gen:
            if msg.get('header', {}).get('msg_type') == 'status' and msg.get('content', {}).get('execution_state') == 'busy': break
    finally:
        await gen.aclose()
        await kc.aclose()
    return ColabJob(self, kc.kernel_id, log_rel, exit_rel)

Launch a 1.5-second detached job on a dedicated kernel:

In [ ]:
job = await rt.launch('''
import time
for i in range(3):
    print(f'step {i}', flush=True)
    time.sleep(0.5)
''')
job

ColabJob('d0c349a7-8b35-40af-bcb7-591ba5a94cc6', log='content/job.log')

`tail` pulls newly appended log bytes over HTTP `Range`. `status` checks the exit file first, then queries `/api/kernels/{kernel_id}` (refreshing the proxy token once on HTTP 404).

In [ ]:
#| export
@patch
async def tail(self:ColabJob)->str:
    "Return newly written log text since the last `tail` call."
    txt, self.offset = await self.rt.tail(self.log, offset=self.offset)
    return txt

@patch
async def status(self:ColabJob)->tuple[str, int|None]:
    "Return `(state, exit_code)` where `state` is `'running'`, `'done'`, `'failed'`, `'idle'`, or `'gone'`."
    try:
        rc = int((await self.rt.download(self.exit_file)).decode().strip())
        return ('done' if rc == 0 else 'failed'), rc
    except FileNotFoundError: pass
    for attempt in range(2):
        async with http_client(self.rt.url, headers=self.rt.headers, timeout=30.0) as hc: resp = await hc.get(f'/api/kernels/{self.kernel_id}')
        if resp.status_code == 404 and attempt == 0:
            await self.rt.refresh()
            continue
        if resp.status_code == 404: return 'gone', None
        state = resp.json().get('execution_state', '')
        return ('running' if state == 'busy' else (state or 'unknown')), None
    return 'gone', None

In [ ]:
st, _ = await job.status()
st

'done'

`wait` polls `tail` and `status` until the exit file appears, invoking an optional `on_log` callback for each new chunk.

In [ ]:
#| export
@patch
async def wait(
    self:ColabJob,
    poll:float=1.0, # Seconds between status polls
    on_log=None, # Optional callback receiving each newly tailed log string
)->int:
    "Poll until the job writes its exit file (or the kernel disappears), returning the exit code."
    while True:
        chunk = await self.tail()
        if chunk and on_log: on_log(chunk)
        st, rc = await self.status()
        if rc is not None:
            rem = await self.tail()
            if rem and on_log: on_log(rem)
            return rc
        if st in ('gone', 'idle'): raise RuntimeError(f'Detached job kernel ended in state {st!r} without writing {self.exit_file}')
        await asyncio.sleep(poll)

In [ ]:
seen = []
rc = await job.wait(poll=0.4, on_log=seen.append)
test_eq(rc, 0)
full_log = ''.join(seen)
test_eq(full_log.splitlines(), ['step 0', 'step 1', 'step 2'])
full_log

'step 0\nstep 1\nstep 2\n'

## Stopping runtimes and context-manager cleanup

Compute units burn whenever a runtime exists, including idle periods. `stop` closes any open local kernel socket and deletes the runtime assignment. Using `async with await c.start('cpu') as rt:` calls `stop` automatically on exit. Attached runtimes (`owned=False`) only close their local socket.

In [ ]:
#| export
@patch
async def stop(
    self:ColabRuntime,
    allow_missing:bool=True, # Ignore already-deleted runtimes?
):
    "Close any local kernel socket and delete this Colab runtime."
    if self._kc is not None:
        try: await self._kc.aclose()
        except Exception: pass
        self._kc = None
    return await self.colab.delete(self.name, allow_missing=allow_missing)

@patch
async def __aenter__(self:ColabRuntime): return self

@patch
async def __aexit__(self:ColabRuntime, *exc):
    if self.owned: await self.stop()
    elif self._kc is not None:
        await self._kc.aclose()
        self._kc = None

Let's stop the CPU runtime we created earlier and confirm no runtimes remain active on the account.

In [ ]:
await rt.stop()
remaining = await c.runtimes()
assert not any(r.name == rt.name for r in remaining)
remaining

[]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()